# 2 · Render - synthetic training images

Renders `IMAGES` images with labels in chunks of `IMAGES_PER_CHUNK` into `renders/`. One mixed run: normal throws,
dice lying close together, **keep and re-roll** (2-4 dice pushed together at the wall) and dice leaning against the
wall or another die (“?”). The shares are set in cell 1.

**Before:** notebook 1 is done (check confirmed), *Runtime → Change runtime type → L4 GPU*.

**If Colab loses the connection:** just run everything again. Finished chunks are skipped and it continues where it stopped.

## 1 · Settings
The trainer has filled in Drive folder and project. Use the same values as in notebook 1 (the preview measured with them).
Changing `SEED` or a `SHARE_…` value later only affects chunks that are not finished yet.

In [ ]:
DRIVE_FOLDER = "__DRIVE__"        # folder in "My Drive"
PROJECT = "__PROJECT__"           # project folder name (under projects/)
NB_NAME, NB_VERSION = "2_render", 1   # version of this notebook (to detect updates)
BLENDER_SERIES = "4.5"            # Blender LTS series; the exact version is recorded the first time
IMAGES = 28000                    # renders in total
IMAGES_PER_CHUNK = 250            # rendered in one go, then saved to Drive
SAMPLES = 32                      # quality (more = nicer, but slower)
SEED = 1
SHARE_CLOSE = 0.35                # throws with dice lying close together
SHARE_LEAN_WALL = 0.2             # images with a die leaning against the wall (“?”)
SHARE_KEEP = 0.2                  # “keep and re-roll”: 2-4 dice pushed together at the wall, the rest thrown
SHARE_LEAN_DIE = 0.05             # images with a die leaning against another die (“?”)

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
P = f"{DRIVE}/projects/{PROJECT}"
SCRIPTS = f"{DRIVE}/scripts"
SCENE = f"{P}/scene"
RENDERS = f"{P}/renders"
PREVIEW = f"{P}/renders_preview"
EXPORT = f"{P}/export"
DET = f"{P}/detector"
CLS = f"{P}/classifier"
ESP_DETECTION = "/content/esp-detection"
print("Project folder:", P)

## 2 · Check the GPU
Expected: **NVIDIA L4** (faster) or **Tesla T4**. If it says "no GPU": *Runtime → Change runtime type → L4 GPU*
(or T4), then start again from the top.

In [ ]:
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("✔ GPU:", gpu) if gpu else print("✘ no GPU! Runtime → Change runtime type → L4 or T4 GPU, then run everything again.")

## 3 · Connect Google Drive and unpack the newest package
Colab asks whether the notebook may access your Drive → **Allow**.

Then the **newest Colab package** in `packages/` is unpacked (the ZIP file from the trainer - upload it to
*My Drive → __DRIVE__ → packages*, do **not** unpack it). Scripts, notebooks, `project.json`, photos, test images and the
camera setup are updated. **Renders and training results are not touched.**
A package is unpacked only once; a newer package replaces photos, test images and camera setup completely.

In [ ]:
import glob, json, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
PACKAGES = f"{DRIVE}/packages"
os.makedirs(PACKAGES, exist_ok=True)
packages = sorted(glob.glob(f"{PACKAGES}/Colab_{PROJECT}_*.zip"))
state_file = f"{DRIVE}/package_state.json"
state = json.load(open(state_file)) if os.path.exists(state_file) else {}
if not packages:
    print(f"⚠ No package “Colab_{PROJECT}_….zip” in {PACKAGES} - create one in the trainer and upload the ZIP file there.")
elif state.get(PROJECT) == os.path.basename(packages[-1]):
    print("✔ newest package is unpacked already:", os.path.basename(packages[-1]))
else:
    new = packages[-1]
    print("unpacking:", os.path.basename(new))
    with zipfile.ZipFile(new) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
        prefix = names[0].split("/")[0]
        # mirror of the laptop: these folders are replaced completely (deleted photos disappear here too)
        for part in ("scripts", "notebooks", f"projects/{PROJECT}/photos", f"projects/{PROJECT}/test_images",
                     f"projects/{PROJECT}/camera_setup"):
            if any(n.startswith(f"{prefix}/{part}/") for n in names) and os.path.isdir(f"{DRIVE}/{part}"):
                shutil.rmtree(f"{DRIVE}/{part}")
        for n in names:
            target = os.path.join(DRIVE, n.split("/", 1)[1])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(n) as src, open(target, "wb") as f:
                shutil.copyfileobj(src, f)
    state[PROJECT] = os.path.basename(new)
    json.dump(state, open(state_file, "w"), indent=1)
    print(f"✔ unpacked ({len(names)} files)")
    try:
        v = json.load(open(f"{DRIVE}/notebooks/version.json")).get(NB_NAME, 0)
        if v > NB_VERSION:
            print(f"⚠ The package contains a NEWER version of this notebook: please open notebooks/{NB_NAME}.ipynb in Drive again.")
    except (OSError, ValueError):
        pass
missing = [p for p in (f"{P}/project.json", f"{SCRIPTS}/render.py") if not os.path.exists(p)]
if missing:
    print("✘ not found:", *missing, sep="\n   ")
else:
    print("✔ project and scripts found")

## 4 · Install Blender (automatic, about 1-3 minutes)
Downloads Blender 4.5 LTS from **download.blender.org** and unpacks it on the Colab disk. The first time the download
is also stored in your Drive (`blender/`) - after that it is faster and it always stays **the same version**.
Colab forgets everything installed in every new session, so this cell runs every time.
At the end it shows whether Blender sees the GPU (expected: **OPTIX** or **CUDA**).

In [ ]:
import json, os, re, shutil, subprocess, urllib.request
CACHE = f"{DRIVE}/blender"
os.makedirs(CACHE, exist_ok=True)
vfile = f"{CACHE}/version.json"
if os.path.exists(vfile):
    version = json.load(open(vfile))["version"]
else:
    html = urllib.request.urlopen(f"https://download.blender.org/release/Blender{BLENDER_SERIES}/").read().decode()
    found = set(re.findall(rf"blender-({re.escape(BLENDER_SERIES)}\.\d+)-linux-x64\.tar\.xz", html))
    version = sorted(found, key=lambda v: int(v.split(".")[-1]))[-1]
    json.dump({"version": version}, open(vfile, "w"))
archive = f"blender-{version}-linux-x64.tar.xz"
if not os.path.exists(f"{CACHE}/{archive}"):
    print("downloading", archive, "…")
    urllib.request.urlretrieve(f"https://download.blender.org/release/Blender{BLENDER_SERIES}/{archive}", f"/content/{archive}")
    shutil.copy(f"/content/{archive}", f"{CACHE}/{archive}")
BLENDER_DIR = f"/content/blender-{version}-linux-x64"
if not os.path.isdir(BLENDER_DIR):
    source = f"/content/{archive}" if os.path.exists(f"/content/{archive}") else f"{CACHE}/{archive}"
    subprocess.run(["tar", "-xf", source, "-C", "/content"], check=True)
subprocess.run("apt-get -qq update > /dev/null; apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
               "libxxf86vm1 libxfixes3 libgl1 libegl1 > /dev/null", shell=True)
BLENDER = f"{BLENDER_DIR}/blender"
print(subprocess.run([BLENDER, "-b", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
test = ("import bpy; p=bpy.context.preferences.addons['cycles'].preferences\n"
        "for t in ('OPTIX','CUDA'):\n"
        "    try:\n        p.compute_device_type=t; p.get_devices()\n"
        "        g=[d.name for d in p.devices if d.type==t]\n"
        "        if g: print('GPU RENDERING:', t, g); break\n"
        "    except TypeError: pass\n"
        "else: print('GPU RENDERING: none - rendering on the CPU (very slow)')")
out = subprocess.run([BLENDER, "-b", "--python-expr", test], capture_output=True, text=True).stdout
print([z for z in out.splitlines() if z.startswith("GPU RENDERING")] or out[-500:])

## 5 · Render the full set (takes hours - see the estimate in notebook 1)
Every finished chunk is saved to Drive right away. **If Colab stops:** open the notebook again, *Runtime → Run all* -
finished chunks are skipped. Tip: keep the browser tab open and do not let the laptop sleep.

In [ ]:
import hashlib, json, os

def atlas_hash():
    return hashlib.sha1(open(f"{SCENE}/atlas.png", "rb").read() + open(f"{SCENE}/scene.json", "rb").read()).hexdigest()

def check_confirmed():
    f = f"{SCENE}/check_ok.json"
    return (os.path.exists(f) and os.path.exists(f"{SCENE}/atlas.png")
            and json.load(open(f)).get("hash") == atlas_hash())

import math, subprocess
assert check_confirmed(), "First notebook 1, cell 7: confirm the check."
chunks = math.ceil(IMAGES / IMAGES_PER_CHUNK)
proc = subprocess.Popen([BLENDER, "-b", "-P", f"{SCRIPTS}/render.py", "--", "--scene", SCENE, "--output", RENDERS,
                         "--chunk-from", "0", "--chunk-to", str(chunks - 1), "--images", str(IMAGES_PER_CHUNK),
                         "--seed", str(SEED), "--samples", str(SAMPLES), "--share-close", str(SHARE_CLOSE), "--share-lean-wall", str(SHARE_LEAN_WALL), "--share-keep", str(SHARE_KEEP), "--share-lean-die", str(SHARE_LEAN_DIE)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for z in proc.stdout:
    if "✔" in z or "⚠" in z or "skipped" in z or "ERROR" in z or "Error" in z or "render device" in z or "image 1/" in z:
        print(z, end="", flush=True)
proc.wait()
print("return code", proc.returncode)

## 6 · Status
Counts the finished chunks and images, shows a contact sheet of the full set and the render statistics
(throw types, dice at the wall, “?” shares). You can run this cell any time (also in a new session after cells 1-3).

In [ ]:
import glob, json, os, subprocess
from IPython.display import Image, display
done = sorted(glob.glob(f"{RENDERS}/chunk_*/done.json"))
n = sum(json.load(open(f))["images"] for f in done)
print(f"{len(done)} chunks done: {n} images (target {IMAGES})")
if done:
    r = subprocess.run(["python", f"{SCRIPTS}/contact_sheet.py", "renders", "--folder", RENDERS, "--output",
                        f"{P}/sheet_renders.jpg", "--count", "16", "--seed", str(len(done))], capture_output=True, text=True)
    print(r.stdout, r.stderr[-1000:])
    display(Image(f"{P}/sheet_renders.jpg", width=1100))
    r = subprocess.run(["python", f"{SCRIPTS}/render_stats.py", "--renders", RENDERS, "--scene", SCENE],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr[-1000:])

## What next?
When the full set is done: `notebooks/3_detector.ipynb`, then `notebooks/4_classifier.ipynb`.

## End the session (optional)
A connected Colab session uses **compute units even when it does nothing**. If you leave while something is still
running: set `END_SESSION = True` below and start this cell with ▶ - it queues up and ends the session when all cells
before it are done. All results are on Drive, nothing is lost.
**Note:** if a cell before it stops with an error, Colab discards the waiting cells - this one too. The session then
keeps running until Colab disconnects it (or you choose *Runtime → Disconnect and delete runtime*).
With *Run all* nothing happens here as long as `END_SESSION = False`.

In [ ]:
END_SESSION = False   # <- set to True and press ▶ if the session should end after the running cells
if END_SESSION:
    from google.colab import runtime
    print("Ending the session - results are on Drive.")
    runtime.unassign()
else:
    print("Session keeps running (END_SESSION = False).")